In [3]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm openpyxl

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 7.6 MB/s eta 0:00:00


In [55]:
!pip install rdkit pandas numpy scikit-learn xgboost openpyxl matplotlib seaborn

In [58]:
"""
QSAR Pipeline for ACE Inhibitor IC50 Prediction
================================================
Target: Angiotensin-Converting Enzyme (ACE) / CHEMBL1808
Input:  Excel file with SMILES + IC50 columns
Output: Trained models + results Excel report
"""

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os
import sys
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem, rdFingerprintGenerator
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import VarianceThreshold

import xgboost as xgb
import openpyxl
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill)
from openpyxl.chart import BarChart, Reference, ScatterChart, Series
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# CONFIG  –  Edit only this block
# ─────────────────────────────────────────────
INPUT_FILE   = r"C:\Users\Sahil\OneDrive\Documents\SAHIL KHAN ONE DRIVE\OneDrive\Desktop\model\ACE INHIBITORS.xlsx"
OUTPUT_FILE  = r"C:\Users\Sahil\OneDrive\Documents\SAHIL KHAN ONE DRIVE\OneDrive\Desktop\model\QSAR_Results.xlsx"
SMILES_COL   = "SMILES"           # column name in your Excel
IC50_COL     = "IC50"   # column name for IC50 numbers (nM)
UNIT_COL     = "Standard Units"   # column with units (nM / uM / M)
TEST_SIZE    = 0.20
RANDOM_STATE = 42
CV_FOLDS     = 5
# ─────────────────────────────────────────────


# ════════════════════════════════════════════════
#  PHASE 1 – DATA LOADING & CLEANING
# ════════════════════════════════════════════════

def load_and_clean(path):
    print("\n[Phase 1] Loading data …")
    df = pd.read_excel(path)
    print(f"  Loaded {len(df)} rows, columns: {list(df.columns)}")

    # keep only IC50
    if "Standard Type" in df.columns:
        df = df[df["Standard Type"] == "IC50"].copy()
        print(f"  Rows with IC50: {len(df)}")

    df = df[[SMILES_COL, IC50_COL]].dropna()
    df.columns = ["SMILES", "IC50_raw"]
    df["IC50_raw"] = pd.to_numeric(df["IC50_raw"], errors="coerce")
    df.dropna(subset=["IC50_raw"], inplace=True)

    # Unit normalisation – assume nM unless file says otherwise
    # If you have a unit column, uncomment and adapt:
    # df["IC50_M"] = df.apply(lambda r: convert_to_M(r), axis=1)
    df["IC50_M"] = df["IC50_raw"] * 1e-9   # nM → M

    # pIC50
    df["pIC50"] = -np.log10(df["IC50_M"].clip(lower=1e-12))

    # Canonicalize SMILES & remove invalids
    canonical, valid = [], []
    for smi in df["SMILES"]:
        mol = Chem.MolFromSmiles(str(smi))
        if mol:
            canonical.append(Chem.MolToSmiles(mol))
            valid.append(True)
        else:
            canonical.append(None)
            valid.append(False)
    df["SMILES"] = canonical
    df = df[valid].drop_duplicates(subset=["SMILES"]).reset_index(drop=True)
    print(f"  Valid & unique molecules: {len(df)}")
    return df


# ════════════════════════════════════════════════
#  PHASE 2 – FEATURE ENGINEERING
# ════════════════════════════════════════════════

DESCRIPTOR_NAMES = [
    "MolWt", "MolLogP", "NumHDonors", "NumHAcceptors",
    "NumRotatableBonds", "TPSA", "NumAromaticRings",
    "RingCount", "FractionCSP3", "NumHeteroatoms",
]

def calc_descriptors(mol):
    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        rdMolDescriptors.CalcNumHBD(mol),
        rdMolDescriptors.CalcNumHBA(mol),
        rdMolDescriptors.CalcNumRotatableBonds(mol),
        Descriptors.TPSA(mol),
        rdMolDescriptors.CalcNumAromaticRings(mol),
        rdMolDescriptors.CalcNumRings(mol),
        rdMolDescriptors.CalcFractionCSP3(mol),
        rdMolDescriptors.CalcNumHeteroatoms(mol),
    ]

def smiles_to_features(smiles_list):
    print("\n[Phase 2] Computing Morgan fingerprints + 2D descriptors …")
    fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        fp  = list(fpgen.GetFingerprintAsNumPy(mol))
        desc = calc_descriptors(mol)
        rows.append(fp + desc)
    cols = [f"fp_{i}" for i in range(2048)] + DESCRIPTOR_NAMES
    X = pd.DataFrame(rows, columns=cols)

    # Remove near-zero-variance fingerprint bits
    sel = VarianceThreshold(threshold=0.01)
    X_sel = sel.fit_transform(X)
    kept_cols = [cols[i] for i in range(len(cols)) if sel.get_support()[i]]
    X = pd.DataFrame(X_sel, columns=kept_cols)
    print(f"  Features after variance filter: {X.shape[1]}")
    return X


# ════════════════════════════════════════════════
#  PHASE 3 – MODEL TRAINING
# ════════════════════════════════════════════════

def train_models(X_train, y_train):
    print("\n[Phase 3] Training models …")
    models = {
        "Random Forest":   RandomForestRegressor(n_estimators=500, max_features="sqrt",
                                                  random_state=RANDOM_STATE, n_jobs=-1),
        "XGBoost":         xgb.XGBRegressor(n_estimators=500, learning_rate=0.05,
                                             max_depth=6, subsample=0.8,
                                             colsample_bytree=0.8,
                                             random_state=RANDOM_STATE,
                                             verbosity=0),
        "Gradient Boost":  GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                      max_depth=5, random_state=RANDOM_STATE),
        "SVR":             SVR(kernel="rbf", C=10, epsilon=0.1),
    }
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_results = {}
    fitted = {}
    for name, model in models.items():
        X_in = X_scaled if name == "SVR" else X_train
        scores = cross_val_score(model, X_in, y_train, cv=kf,
                                 scoring="r2", n_jobs=-1)
        cv_results[name] = {"mean_R2": scores.mean(), "std_R2": scores.std()}
        model.fit(X_in, y_train)
        fitted[name] = model
        print(f"  {name:<20} CV R² = {scores.mean():.3f} ± {scores.std():.3f}")

    return fitted, scaler, cv_results


# ════════════════════════════════════════════════
#  Y-RANDOMIZATION SANITY CHECK
# ════════════════════════════════════════════════

def y_randomization(X_train, y_train, n_iter=10):
    print("\n[Phase 4] Y-Randomization check …")
    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scrambled_r2s = []
    for _ in range(n_iter):
        y_shuf = y_train.sample(frac=1, random_state=np.random.randint(9999)).values
        s = cross_val_score(rf, X_train, y_shuf, cv=kf, scoring="r2").mean()
        scrambled_r2s.append(s)
    mean_rand = np.mean(scrambled_r2s)
    print(f"  Mean scrambled R² = {mean_rand:.3f}  (should be near 0)")
    return scrambled_r2s, mean_rand


# ════════════════════════════════════════════════
#  PHASE 5 – EVALUATION
# ════════════════════════════════════════════════

def evaluate(fitted, scaler, X_test, y_test):
    print("\n[Phase 5] Test-set evaluation …")
    results = {}
    X_scaled = scaler.transform(X_test)
    for name, model in fitted.items():
        X_in = X_scaled if name == "SVR" else X_test
        y_pred = model.predict(X_in)
        r2   = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        results[name] = {"R2": r2, "RMSE": rmse, "MAE": mae, "y_pred": y_pred}
        print(f"  {name:<20} R²={r2:.3f}  RMSE={rmse:.3f}  MAE={mae:.3f}")
    return results


# ════════════════════════════════════════════════
#  EXCEL REPORT
# ════════════════════════════════════════════════

HEADER_FILL  = PatternFill("solid", fgColor="2E4057")
HEADER_FONT  = Font(bold=True, color="FFFFFF", name="Arial", size=11)
ALT_FILL     = PatternFill("solid", fgColor="EBF0F7")
GOOD_FILL    = PatternFill("solid", fgColor="C6EFCE")
BAD_FILL     = PatternFill("solid", fgColor="FFC7CE")
THIN_BORDER  = Border(
    left=Side(style="thin"),   right=Side(style="thin"),
    top=Side(style="thin"),    bottom=Side(style="thin"),
)

def style_header_row(ws, row, ncols, start_col=1):
    for c in range(start_col, start_col + ncols):
        cell = ws.cell(row=row, column=c)
        cell.fill = HEADER_FILL
        cell.font = HEADER_FONT
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = THIN_BORDER

def build_report(df_clean, cv_results, test_results, scrambled_r2s,
                 rand_mean, y_test, path):
    wb = openpyxl.Workbook()

    # ── Sheet 1: Summary Dashboard ──────────────────────────────────────
    ws = wb.active
    ws.title = "Summary Dashboard"
    ws.sheet_view.showGridLines = False
    ws.column_dimensions["A"].width = 24
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 18
    ws.column_dimensions["D"].width = 18

    ws["A1"] = "QSAR Model Report – ACE Inhibitors"
    ws["A1"].font = Font(bold=True, size=16, color="2E4057", name="Arial")
    ws["A1"].alignment = Alignment(horizontal="center")
    ws.merge_cells("A1:D1")

    ws["A2"] = f"Dataset: {len(df_clean)} clean molecules   |   pIC50 range: {df_clean['pIC50'].min():.2f} – {df_clean['pIC50'].max():.2f}"
    ws["A2"].font = Font(italic=True, color="555555", name="Arial", size=10)
    ws.merge_cells("A2:D2")

    # CV Results table
    ws["A4"] = "Cross-Validation (Training Set)"
    ws["A4"].font = Font(bold=True, size=12, color="2E4057", name="Arial")
    headers_cv = ["Model", "CV R² (mean)", "CV R² (std)"]
    for i, h in enumerate(headers_cv, 1):
        ws.cell(row=5, column=i).value = h
    style_header_row(ws, 5, 3)
    for r, (name, res) in enumerate(cv_results.items(), 6):
        ws.cell(row=r, column=1).value = name
        ws.cell(row=r, column=2).value = round(res["mean_R2"], 3)
        ws.cell(row=r, column=3).value = round(res["std_R2"],  3)
        fill = GOOD_FILL if res["mean_R2"] >= 0.6 else BAD_FILL
        for c in range(1, 4):
            cell = ws.cell(row=r, column=c)
            cell.fill = fill if c > 1 else ALT_FILL
            cell.border = THIN_BORDER
            cell.alignment = Alignment(horizontal="center")
            cell.font = Font(name="Arial", size=10)

    # Test Results table
    row_start = 6 + len(cv_results) + 2
    ws.cell(row=row_start, column=1).value = "Test-Set Performance (Unseen Data)"
    ws.cell(row=row_start, column=1).font = Font(bold=True, size=12, color="2E4057", name="Arial")
    headers_t = ["Model", "R²", "RMSE (pIC50)", "MAE (pIC50)"]
    for i, h in enumerate(headers_t, 1):
        ws.cell(row=row_start+1, column=i).value = h
    style_header_row(ws, row_start+1, 4)
    for r, (name, res) in enumerate(test_results.items(), row_start+2):
        ws.cell(row=r, column=1).value = name
        ws.cell(row=r, column=2).value = round(res["R2"],   3)
        ws.cell(row=r, column=3).value = round(res["RMSE"], 3)
        ws.cell(row=r, column=4).value = round(res["MAE"],  3)
        fill = GOOD_FILL if res["R2"] >= 0.6 else BAD_FILL
        for c in range(1, 5):
            cell = ws.cell(row=r, column=c)
            cell.fill = fill if c > 1 else ALT_FILL
            cell.border = THIN_BORDER
            cell.alignment = Alignment(horizontal="center")
            cell.font = Font(name="Arial", size=10)

    # Y-Randomization
    yr_row = row_start + len(test_results) + 4
    ws.cell(row=yr_row, column=1).value = "Y-Randomization Check"
    ws.cell(row=yr_row, column=1).font = Font(bold=True, size=12, color="2E4057", name="Arial")
    ws.cell(row=yr_row+1, column=1).value = "Mean Scrambled R²"
    ws.cell(row=yr_row+1, column=2).value = round(rand_mean, 3)
    verdict = "✓ PASS – Model is learning real patterns" if rand_mean < 0.2 else "✗ FAIL – Possible overfitting"
    ws.cell(row=yr_row+1, column=3).value = verdict
    ws.cell(row=yr_row+1, column=3).font = Font(
        bold=True, color="006100" if rand_mean < 0.2 else "9C0006", name="Arial")

    # ── Sheet 2: Processed Data ──────────────────────────────────────────
    ws2 = wb.create_sheet("Processed Data")
    ws2.sheet_view.showGridLines = False
    cols2 = ["SMILES", "IC50_raw (nM)", "IC50_M (M)", "pIC50"]
    for i, h in enumerate(cols2, 1):
        ws2.cell(row=1, column=i).value = h
    style_header_row(ws2, 1, 4)
    ws2.column_dimensions["A"].width = 55
    for col in ["B", "C", "D"]:
        ws2.column_dimensions[col].width = 18
    for r, row in enumerate(df_clean.itertuples(index=False), 2):
        ws2.cell(row=r, column=1).value = row.SMILES
        ws2.cell(row=r, column=2).value = round(row.IC50_raw, 4)
        ws2.cell(row=r, column=3).value = float(f"{row.IC50_M:.4e}")
        ws2.cell(row=r, column=4).value = round(row.pIC50, 4)
        fill = ALT_FILL if r % 2 == 0 else PatternFill()
        for c in range(1, 5):
            cell = ws2.cell(row=r, column=c)
            cell.fill = fill
            cell.border = THIN_BORDER
            cell.font = Font(name="Arial", size=9)
            cell.alignment = Alignment(horizontal="left" if c == 1 else "center")

    # ── Sheet 3: Predicted vs Actual ─────────────────────────────────────
    ws3 = wb.create_sheet("Predictions")
    ws3.sheet_view.showGridLines = False
    best_model = max(test_results, key=lambda k: test_results[k]["R2"])
    ws3["A1"] = f"Best Model: {best_model}"
    ws3["A1"].font = Font(bold=True, size=13, color="2E4057", name="Arial")
    headers_p = ["#", "Actual pIC50", "Predicted pIC50", "Error", "|Error|"]
    for i, h in enumerate(headers_p, 1):
        ws3.cell(row=2, column=i).value = h
    style_header_row(ws3, 2, 5)
    for col in ["A", "B", "C", "D", "E"]:
        ws3.column_dimensions[col].width = 18
    y_pred_best = test_results[best_model]["y_pred"]
    for r, (actual, pred) in enumerate(zip(y_test, y_pred_best), 3):
        err = pred - actual
        ws3.cell(row=r, column=1).value = r - 2
        ws3.cell(row=r, column=2).value = round(float(actual), 3)
        ws3.cell(row=r, column=3).value = round(float(pred),   3)
        ws3.cell(row=r, column=4).value = round(float(err),    3)
        ws3.cell(row=r, column=5).value = round(abs(float(err)), 3)
        fill = ALT_FILL if r % 2 == 0 else PatternFill()
        for c in range(1, 6):
            cell = ws3.cell(row=r, column=c)
            cell.fill = fill
            cell.border = THIN_BORDER
            cell.font = Font(name="Arial", size=9)
            cell.alignment = Alignment(horizontal="center")

    # scatter chart on ws3
    n_test = len(y_pred_best)
    chart = ScatterChart()
    chart.title = "Actual vs Predicted pIC50"
    chart.style = 10
    chart.x_axis.title = "Actual pIC50"
    chart.y_axis.title = "Predicted pIC50"
    xvals = Reference(ws3, min_col=2, min_row=3, max_row=2+n_test)
    yvals = Reference(ws3, min_col=3, min_row=3, max_row=2+n_test)
    series = Series(yvals, xvals, title=best_model)
    series.marker.symbol = "circle"
    series.marker.size = 5
    series.graphicalProperties.line.noFill = True
    chart.series.append(series)
    chart.width = 18; chart.height = 14
    ws3.add_chart(chart, "G3")

    # ── Sheet 4: pIC50 Distribution ──────────────────────────────────────
    ws4 = wb.create_sheet("pIC50 Distribution")
    ws4.sheet_view.showGridLines = False
    bins = np.arange(int(df_clean["pIC50"].min()), int(df_clean["pIC50"].max())+2, 1)
    counts, edges = np.histogram(df_clean["pIC50"], bins=bins)
    ws4["A1"] = "pIC50 Distribution"
    ws4["A1"].font = Font(bold=True, size=13, color="2E4057", name="Arial")
    ws4["A2"] = "Bin"; ws4["B2"] = "Count"
    style_header_row(ws4, 2, 2)
    for i, (lo, cnt) in enumerate(zip(edges[:-1], counts), 3):
        ws4.cell(row=i, column=1).value = f"{lo:.0f}–{lo+1:.0f}"
        ws4.cell(row=i, column=2).value = int(cnt)
        for c in range(1, 3):
            ws4.cell(row=i, column=c).border = THIN_BORDER
            ws4.cell(row=i, column=c).font = Font(name="Arial", size=10)
            ws4.cell(row=i, column=c).alignment = Alignment(horizontal="center")
    bar = BarChart()
    bar.type = "col"; bar.style = 10
    bar.title = "Distribution of pIC50 Values"
    bar.y_axis.title = "Count"
    bar.x_axis.title = "pIC50 Bin"
    data_ref  = Reference(ws4, min_col=2, min_row=2, max_row=2+len(counts))
    cats_ref  = Reference(ws4, min_col=1, min_row=3, max_row=2+len(counts))
    bar.add_data(data_ref, titles_from_data=True)
    bar.set_categories(cats_ref)
    bar.width = 20; bar.height = 14
    ws4.add_chart(bar, "D3")

    # ── Sheet 5: Model Comparison Bar Chart ─────────────────────────────
    ws5 = wb.create_sheet("Model Comparison")
    ws5.sheet_view.showGridLines = False
    ws5["A1"] = "Model Comparison"; ws5["A1"].font = Font(bold=True, size=13, color="2E4057", name="Arial")
    headers_mc = ["Model", "CV R²", "Test R²", "Test RMSE", "Test MAE"]
    for i, h in enumerate(headers_mc, 1):
        ws5.cell(row=2, column=i).value = h
    style_header_row(ws5, 2, 5)
    for col in ["A","B","C","D","E"]:
        ws5.column_dimensions[col].width = 20
    for r, name in enumerate(cv_results.keys(), 3):
        ws5.cell(row=r, column=1).value = name
        ws5.cell(row=r, column=2).value = round(cv_results[name]["mean_R2"], 3)
        ws5.cell(row=r, column=3).value = round(test_results[name]["R2"],   3)
        ws5.cell(row=r, column=4).value = round(test_results[name]["RMSE"], 3)
        ws5.cell(row=r, column=5).value = round(test_results[name]["MAE"],  3)
        for c in range(1, 6):
            cell = ws5.cell(row=r, column=c)
            cell.border = THIN_BORDER
            cell.font = Font(name="Arial", size=10)
            cell.alignment = Alignment(horizontal="center")
            cell.fill = ALT_FILL if r % 2 == 0 else PatternFill()
    n_models = len(cv_results)
    bar2 = BarChart()
    bar2.type = "col"; bar2.style = 10
    bar2.title = "Test R² by Model"
    bar2.y_axis.title = "R²"; bar2.y_axis.numFmt = "0.000"
    bar2.y_axis.scaling.min = 0; bar2.y_axis.scaling.max = 1
    data2 = Reference(ws5, min_col=3, max_col=3, min_row=2, max_row=2+n_models)
    cats2 = Reference(ws5, min_col=1, min_row=3, max_row=2+n_models)
    bar2.add_data(data2, titles_from_data=True)
    bar2.set_categories(cats2)
    bar2.width = 20; bar2.height = 14
    ws5.add_chart(bar2, "G3")

    wb.save(path)
    print(f"\n✅ Report saved → {path}")


# ════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════

def main():
    # Phase 1
    df = load_and_clean(INPUT_FILE)

    # Phase 2
    X = smiles_to_features(df["SMILES"].tolist())
    y = df["pIC50"].values

    # Train / Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    print(f"\n  Train: {len(X_train)} | Test: {len(X_test)}")

    # Phase 3
    fitted, scaler, cv_results = train_models(X_train, y_train)

    # Phase 4 – Y-randomization
    scrambled, rand_mean = y_randomization(X_train, pd.Series(y_train))

    # Phase 5
    test_results = evaluate(fitted, scaler, X_test, y_test)

    # Build Excel report
    print("\n[Report] Writing Excel report …")
    build_report(df, cv_results, test_results, scrambled,
                 rand_mean, y_test, OUTPUT_FILE)

    # Print best model
    best = max(test_results, key=lambda k: test_results[k]["R2"])
    print(f"\n🏆 Best model: {best}  (Test R² = {test_results[best]['R2']:.3f})")


if __name__ == "__main__":
    main()


[Phase 1] Loading data …
  Loaded 2082 rows, columns: ['SMILES', 'Standard Type', 'Standard Relation', 'IC50', 'Standard Units']
  Rows with IC50: 2082
  Valid & unique molecules: 1667

[Phase 2] Computing Morgan fingerprints + 2D descriptors …
  Features after variance filter: 617

  Train: 1333 | Test: 334

[Phase 3] Training models …
  Random Forest        CV R² = 0.670 ± 0.024
  XGBoost              CV R² = 0.665 ± 0.027
  Gradient Boost       CV R² = 0.675 ± 0.013
  SVR                  CV R² = 0.665 ± 0.027

[Phase 4] Y-Randomization check …
  Mean scrambled R² = -0.179  (should be near 0)

[Phase 5] Test-set evaluation …
  Random Forest        R²=0.700  RMSE=0.989  MAE=0.705
  XGBoost              R²=0.707  RMSE=0.977  MAE=0.692
  Gradient Boost       R²=0.700  RMSE=0.988  MAE=0.705
  SVR                  R²=0.696  RMSE=0.996  MAE=0.720

[Report] Writing Excel report …

✅ Report saved → C:\Users\Sahil\OneDrive\Documents\SAHIL KHAN ONE DRIVE\OneDrive\Desktop\model\QSAR_Results.x